# MADRL CityLearn v3 — Tutorial Completo (Google Colab · A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/master/CityLearn/examples/madrl_citylearn_v3_tutorial.ipynb)

**Proyecto:** Multi-Agente de Aprendizaje por Refuerzo Profundo para gestion coordinada
de flexibilidad energetica, emisiones de CO2 y eficiencia economica en comunidades inteligentes.

**Caso de estudio:** 17 edificios reales de Iquitos, Peru · Dataset 2023-2025 · 26 304 pasos horarios.

| Parametro | Valor |
|---|---|
| Algoritmos | HAPPO · MASAC · MATD3 · MAAC |
| Escenarios | E1 (Flexibilidad) · E2 (CO2) · E3 (Costos) |
| Episodios | 75 por corrida · 8 760 pasos/episodio |
| Total steps | 657 000 por corrida · 7 884 000 en total |
| GPU objetivo | A100 40 GB o 80 GB (Colab Pro/Pro+) |
| Ejecucion | Secuencial, recuperable, con monitor visible y reintento OOM |

> **Requisito:** Seleccionar A100 en *Runtime -> Change runtime type -> A100 GPU*. El notebook falla temprano si Colab entrega otra GPU.

### Fuentes cientificas y de tesis usadas para el diseno

- CityLearn estandariza la evaluacion RL/MARL para demanda respuesta urbana: https://arxiv.org/abs/2012.10504
- CityLearn v2 y CityLearn Challenge documentan KPIs de flexibilidad, carbono y costo: https://escholarship.org/content/qt5t48x8xk/qt5t48x8xk.pdf y https://proceedings.mlr.press/v220/nweye23a.html
- HAPPO/HATRPO justifica actualizacion secuencial y trust region en MARL: https://openreview.net/forum?id=EcGGFkNTxdJ
- MAAC usa criticos centralizados con atencion para escalar agentes: https://proceedings.mlr.press/v97/iqbal19a.html
- MATD3 reduce sobreestimacion mediante doble critico centralizado: https://arxiv.org/abs/1910.01465
- MASAC se apoya en SAC y mezcla QMIX/CTDE: https://arxiv.org/abs/1812.05905 y https://arxiv.org/abs/1803.11485
- PyTorch CUDA y reproducibilidad: https://docs.pytorch.org/docs/stable/notes/cuda.html y https://docs.pytorch.org/docs/stable/notes/randomness.html
- Colab no garantiza tipo de GPU ni duracion; por eso se requiere checkpoint/estado recuperable: https://research.google.com/colaboratory/faq.html
- A100 40/80 GB, HBM y TF32: https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/nvidia-a100-datasheet-us-nvidia-1758950-r4-web.pdf
- Tesis consultadas sobre RL/MARL energetico: Ross May PhD Dalarna 2023, Oxford residential flexibility thesis, Politecnico di Torino MARL building-energy thesis.


## Sección 1: Configuración inicial

In [ ]:
# ── 1.1  Verificar GPU ──────────────────────────────────────────────────────
import subprocess, os, sys

res = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", res.stdout.strip())

import torch
print(f"PyTorch {torch.__version__}  |  CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Dispositivo: {name}  |  VRAM: {mem:.1f} GiB")
    if "A100" in name:
        print("✅ A100 detectado — parámetros A100 activos")
    else:
        print(f"⚠️  GPU detectada: {name} — parámetros A100 pueden ser excesivos")
else:
    raise RuntimeError("❌ No hay GPU disponible. Habilita la GPU A100 en Runtime settings.")


In [ ]:
# ── 1.2  Clonar repositorio con submodulos ──────────────────────────────────
import os, subprocess

REPO_URL = 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'
REPO     = '/content/MADRLCitytleranflexresdr'

if not os.path.exists(f'{REPO}/.git'):
    print('Clonando repositorio con submodulos...')
    subprocess.check_call(['git', 'clone', '--recurse-submodules', '--depth', '1', REPO_URL, REPO])
else:
    print('Repositorio ya existe; actualizando submodulos...')
    subprocess.check_call(['git', '-C', REPO, 'submodule', 'update', '--init', '--recursive'])

os.chdir(REPO)
print(f'\nDirectorio de trabajo: {os.getcwd()}')


In [ ]:
# ── 1.3  Instalar dependencias del proyecto de forma reproducible ───────────
# No se instalan como paquetes los backends que no tienen setup.py/pyproject
# (external/MARL/src y external/MAAC). Esos se exponen via sys.path.
import os, sys, subprocess

os.chdir('/content/MADRLCitytleranflexresdr')

def pip_install(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', *args]
    print(' '.join(cmd))
    subprocess.check_call(cmd)

pip_install('-q', '-e', 'CityLearn/')
pip_install('-q', '-e', 'external/HARL/')
pip_install('-q', '-e', 'external/off-policy/')
pip_install('-q', 'scipy', 'pandas', 'matplotlib', 'seaborn', 'tensorboard', 'tensorboardX', 'setproctitle', 'simplejson')

print('\nDependencias instaladas. MASAC y MAAC se cargan por sys.path, no por pip editable.')


In [ ]:
# ── 1.4  Configurar sys.path, CUDA y smoke imports ──────────────────────────
import os, sys, importlib, json
from pathlib import Path

REPO = '/content/MADRLCitytleranflexresdr'
_paths = [
    REPO,
    f'{REPO}/CityLearn',
    f'{REPO}/CityLearn/scripts',
    f'{REPO}/external/HARL',
    f'{REPO}/external/MARL/src',
    f'{REPO}/external/off-policy',
    f'{REPO}/external/MAAC',
]
for p in reversed(_paths):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ['PYTHONPATH'] = ':'.join(_paths + [os.environ.get('PYTHONPATH', '')])
os.environ['CITYLEARN_PROJECT_ROOT'] = REPO
os.environ.setdefault('CUDA_DEVICE_ORDER', 'PCI_BUS_ID')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('PYTHONHASHSEED', '0')

required_paths = [Path(p) for p in _paths]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f'Rutas requeridas no encontradas: {missing}')

modules = [
    'torch',
    'numpy',
    'pandas',
    'citylearn',
    'citylearn.v3.environment',
    'harl',
    'runner_msac',
    'offpolicy',
    'algorithms.attention_sac',
]
smoke = {}
for name in modules:
    try:
        importlib.import_module(name)
        smoke[name] = 'ok'
    except Exception as exc:
        smoke[name] = f'FAILED: {exc}'

print(json.dumps(smoke, indent=2))
failures = {k: v for k, v in smoke.items() if v.startswith('FAILED')}
if failures:
    raise RuntimeError(f'Smoke imports fallaron: {failures}')

print('sys.path, CUDA env y smoke imports configurados.')


### Persistencia obligatoria en Google Drive

Para 75 episodios en Colab, los artefactos deben persistir fuera de `/content`. La siguiente celda monta Drive por defecto. Si no estas en Colab, usa el fallback local dentro del repo.


In [ ]:
# ── 1.5  Montar Google Drive para checkpoints y reanudacion ─────────────────
import os

USE_GOOGLE_DRIVE = True
GDRIVE_ROOT = None

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        GDRIVE_ROOT = '/content/drive/MyDrive/MADRL_CityLearn_v3'
        os.makedirs(GDRIVE_ROOT, exist_ok=True)
        print('Google Drive montado:', GDRIVE_ROOT)
    except Exception as exc:
        print('Drive no disponible; usando outputs local del runtime:', exc)
        GDRIVE_ROOT = None


## Sección 2: Configuración del proyecto

In [ ]:
# ── 2.1  Rutas, timestamp y directorio de salida recuperable ────────────────
import os, sys
from datetime import datetime
from pathlib import Path

REPO        = '/content/MADRLCitytleranflexresdr'
TIMESTAMP   = datetime.now().strftime('%Y%m%d_%H%M%S')
SCHEMA_PATH = f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json'
PYTHON      = sys.executable

BASE_OUTPUT_PARENT = GDRIVE_ROOT if GDRIVE_ROOT else f'{REPO}/outputs'
OUTPUT_ROOT = f'{BASE_OUTPUT_PARENT}/colab_madrl_a100_{TIMESTAMP}'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(f'{REPO}/outputs', exist_ok=True)

# El monitor Colab y el monitor oficial buscan estas rutas.
for latest_name in ['latest_colab_output_root.txt', 'latest_visible_training_output_root.txt']:
    with open(f'{REPO}/outputs/{latest_name}', 'w') as _f:
        _f.write(OUTPUT_ROOT)

assert os.path.exists(SCHEMA_PATH), f'Schema no encontrado: {SCHEMA_PATH}'

print(f'TIMESTAMP   : {TIMESTAMP}')
print(f'OUTPUT_ROOT : {OUTPUT_ROOT}')
print(f'SCHEMA_PATH : {SCHEMA_PATH}  OK')


## Sección 3: Dataset Iquitos 2023-2025

**17 edificios reales** · 26 304 pasos horarios · 222 CSV sin NaN/Inf

| Recurso | Detalle |
|---|---|
| Período | 2023-2025 · año horario completo |
| BESS total | 26 266 kWh / 6 648 kW |
| PV total | 48 790 kWp (PVGIS TMY/pvlib) |
| EV chargers | 185 tomas · 96 equipos · 1 850 EVs en pool |
| V2G | 31 tomas de camiones (B01 Electro Oriente) |
| Intensidad carbono | 0.671-0.790 kgCO₂/kWh (MINAM RAGEI 2019) |
| Tarifa punta (18-22h) | 0.38 USD/kWh · fuera punta: 0.26 USD/kWh |


In [ ]:
# ── 3.1  Verificar estructura del dataset ────────────────────────────────────
import json, os, pandas as pd

with open(SCHEMA_PATH) as f:
    schema = json.load(f)

buildings = schema.get("buildings", {})
print(f"Edificios: {len(buildings)}")
print(f"Pasos de simulación: {schema.get('simulation_end_time_step', 0) + 1}")
print(f"Agente central: {schema.get('central_agent', False)}")

# Mostrar primeros 5 edificios
for i, (name, bld) in enumerate(buildings.items()):
    if i >= 5:
        print(f"  ... y {len(buildings)-5} edificios más")
        break
    ev   = len(bld.get("electric_vehicle_chargers", []))
    bess = bld.get("electrical_storage", {}).get("capacity", "N/A")
    pv   = bld.get("pv", {}).get("nominal_power", "N/A")
    print(f"  {name}: EV={ev} tomas | BESS={bess} kWh | PV={pv} kWp")

# Verificar primer CSV
first_bld = list(buildings.keys())[0]
csv_rel = buildings[first_bld].get("energy_simulation", "")
csv_full = f"{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/{csv_rel}"
df = pd.read_csv(csv_full)
print(f"\nCSV {first_bld}: shape={df.shape} — filas ok: {len(df)==26304}")


## Sección 4: Entorno Dec-POMDP — 17 agentes

**Dec-POMDP:** cada edificio es un agente con observación parcial local.
**CTDE:** el crítico usa el estado global durante entrenamiento; la ejecución es completamente local.

```
Observación local oᵢ(t) ≈ 40 dimensiones
  ├── Tiempo (mes, hora, tipo_día)
  ├── Física edificio (NSL, DHW, cooling, T_interior)
  ├── BESS (SOC, acción previa)
  ├── EV (SOC_k, salida_k, SOC_req_k)
  └── Señales globales (carbono, precio, GHI, T_amb)

Acción local aᵢ(t): [BESS_charge, EV_charge, Lavadora_on_off]
```


In [ ]:
# ── 4.1  Crear entorno smoke-test (4 pasos) y describir agentes ─────────────
from citylearn.v3.environment import make_citylearn_v3_project_env, describe_environment

env = make_citylearn_v3_project_env(
    scenario="E1",
    seed=0,
    episode_time_steps=4,
    reward_aggregation="team_mean",
    normalize_observations=True,
    madrl_algorithm="MATD3",
    use_citylearn_v3_reward=True,
)
desc = describe_environment(env)
env.close()

obs_dims = list(desc.get("observation_dims", {}).values())
act_dims = list(desc.get("action_dims",      {}).values())

print(f"Num agentes  : {desc['num_agents']}")
print(f"Obs dim      : {obs_dims[0] if obs_dims else '?'}  (por agente)")
print(f"Action dim   : {act_dims[0] if act_dims else '?'}  (por agente)")
print(f"Reward func  : {desc.get('reward_function', 'N/A')}")
print(f"Reward aggr  : {desc.get('reward_aggregation', 'N/A')}")
print(f"Escenario    : E1 (Flexibilidad energética)")
print("\n✅ Entorno Dec-POMDP verificado.")


## Sección 5: Función de recompensa multiobjetivo

### Componentes (v4)

| Componente | Descripción |
|---|---|
| **Flexibilidad** | peak_penalty + ramping_penalty + load_factor + ev_service |
| **CO₂** | carbon_emissions × carbon_intensity |
| **Costo** | electricity_cost × price_signal |
| **EV urgency** | SOC_deficit × 1/horas_hasta_salida |
| **BESS degradación** | C-rate penalty Arrhenius LiFePO₄ (v4) |

### Pesos por escenario

| Escenario | flex | carbon | cost |
|:---:|:---:|:---:|:---:|
| **E1** | **0.70** | 0.15 | 0.15 |
| **E2** | 0.15 | **0.70** | 0.15 |
| **E3** | 0.25 | 0.15 | **0.60** |

### Recompensa mixta CTDE (team_ratio = 0.70)
```
r_i_mix = 0.30 × r_i_local  +  0.70 × mean(r₁,...,r₁₇)
```


In [ ]:
# ── 5.1  Visualizar pesos de recompensa por escenario ────────────────────────
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, os

WEIGHTS = {
    "E1": {"Flexibilidad": 0.70, "CO₂": 0.15, "Costo": 0.15},
    "E2": {"Flexibilidad": 0.15, "CO₂": 0.70, "Costo": 0.15},
    "E3": {"Flexibilidad": 0.25, "CO₂": 0.15, "Costo": 0.60},
}
COLORS = ["#3b82f6", "#22c55e", "#f59e0b"]
LABELS = list(WEIGHTS["E1"].keys())

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
fig.suptitle("Pesos de recompensa por escenario (CityLearnV3MADRLRewardFunction v4)",
             fontsize=13, fontweight="bold")
for ax, (sc, wts), in zip(axes, WEIGHTS.items()):
    vals = list(wts.values())
    bars = ax.bar(LABELS, vals, color=COLORS, edgecolor="white", linewidth=1.5, width=0.55)
    ax.set_title(f"Escenario {sc}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 0.85)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
    ax.set_facecolor("#f8fafc")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{v:.2f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
os.makedirs(f"{OUTPUT_ROOT}/figures", exist_ok=True)
plt.savefig(f"{OUTPUT_ROOT}/figures/reward_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅  Figura: {OUTPUT_ROOT}/figures/reward_weights.png")


## Seccion 6: Hiperparametros (A100 estable · 75 episodios)

La configuracion usa un perfil A100 secuencial y recuperable. La meta no es "nunca fallar" —Colab no garantiza recursos— sino fallar temprano, guardar estado, reanudar y reducir riesgo de OOM.

| Parametro | Valor estable A100 |
|---|:---:|
| Episodios | 75 |
| Pasos/episodio | 8 760 |
| Torch threads | 2 |
| Artifact profile | efficient |
| Trace interval | 24 pasos |
| Live progress | cada 1 000 pasos |
| CUDA memory fraction | 0.92 |
| Ejecucion | secuencial por job |
| Reanudacion | `--skip-completed` |
| OOM retry | activo para MASAC/MATD3/MAAC |
| HAPPO hidden_size | 384 |
| MASAC buffer_size / critic_batch | 20 / 64, retry 10 / 32 |
| MATD3 batch / buffer | 512 / 6000, retry 256 / 4096 |
| MAAC batch / buffer | 512 / 100000, retry 256 / 50000 |


In [ ]:
# ── 6.1  Configuracion central de entrenamiento A100 ───────────────────────
import os, sys, subprocess, json, time
from pathlib import Path

REPO        = '/content/MADRLCitytleranflexresdr'
PYTHON      = sys.executable
SCHEMA_PATH = f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json'
LAUNCHER    = f'{REPO}/CityLearn/scripts/colab_a100_official_launcher.py'
MONITOR     = f'{REPO}/CityLearn/scripts/colab_a100_live_monitor.py'

# Cambiar a True solo si se desea una prueba corta de infraestructura.
QUICK_TEST = False
EPISODES        = 3 if QUICK_TEST else 75
EPISODE_STEPS   = 8760
NUM_ENV_STEPS   = EPISODES * EPISODE_STEPS
SEED            = 0

TORCH_THREADS        = 2
LIVE_PROGRESS_INT    = 1000
LIVE_HEARTBEAT_SEC   = 30
ARTIFACT_PROFILE     = 'efficient'
TRACE_INTERVAL       = 24
TRACE_DETAIL         = 'compact'
GPU_PROFILE          = 'aws'
CUDA_MEMORY_FRACTION = 0.92

SCENARIOS  = ['E1', 'E2', 'E3']
ALGORITHMS = ['happo', 'masac', 'matd3', 'maac']

mode = 'QUICK_TEST (3 ep)' if QUICK_TEST else 'FULL TRAINING (75 ep)'
print(f'Modo          : {mode}')
print(f'Episodios     : {EPISODES} x {EPISODE_STEPS} pasos = {NUM_ENV_STEPS:,} pasos/corrida')
print(f'Corridas total: {len(SCENARIOS) * len(ALGORITHMS)} ({len(ALGORITHMS)} algos x {len(SCENARIOS)} escenarios)')
print(f'Output root   : {OUTPUT_ROOT}')
print(f'Launcher      : {LAUNCHER}')


## Seccion 7: Lanzamiento oficial recuperable

El entrenamiento ya no se lanza con cuatro bloques manuales. Se usa un orquestador unico que genera `official_full_status.json`, `official_full_manifest.json`, logs por job, checkpoint/resume y monitor visible.


In [ ]:
# ── 7.0  Helpers de ejecucion y monitor ─────────────────────────────────────
import subprocess, sys, os, json
from pathlib import Path


def run_cmd(cmd, *, cwd=REPO, check=True):
    print('\n' + '=' * 80)
    print(' '.join(str(c) for c in cmd))
    print('=' * 80)
    proc = subprocess.run(cmd, cwd=cwd, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Comando fallo con exit={proc.returncode}')
    return proc.returncode


def launcher_base_args():
    return [
        PYTHON, '-B', LAUNCHER,
        '--scenario', 'ALL',
        '--seed', str(SEED),
        '--episode-time-steps', str(EPISODE_STEPS),
        '--episodes', str(EPISODES),
        '--schema-path', SCHEMA_PATH,
        '--output-root', OUTPUT_ROOT,
        '--torch-threads', str(TORCH_THREADS),
        '--live-progress-interval', str(LIVE_PROGRESS_INT),
        '--live-heartbeat-seconds', str(LIVE_HEARTBEAT_SEC),
        '--artifact-profile', ARTIFACT_PROFILE,
        '--trace-record-interval', str(TRACE_INTERVAL),
        '--trace-detail', TRACE_DETAIL,
        '--gpu-profile', GPU_PROFILE,
        '--cuda-memory-fraction', str(CUDA_MEMORY_FRACTION),
        '--require-a100',
        '--smoke-imports',
        '--oom-retry',
        '--live-monitor',
        '--monitor-interval', '30',
    ]


def monitor_once():
    return run_cmd([PYTHON, '-B', MONITOR, '--output-root', OUTPUT_ROOT, '--once', '--log-tail', '18'], check=False)


### 7.1 Preflight y dry-run obligatorio

Esta celda valida A100, CUDA, imports, rutas, manifest y los 12 comandos planificados sin entrenar. Si falla aqui, no ejecutes el entrenamiento completo.


In [ ]:
# ── 7.1  Preflight A100 + dry-run oficial ───────────────────────────────────
dry_run_cmd = launcher_base_args() + ['--dry-run', '--skip-completed']
run_cmd(dry_run_cmd)
monitor_once()

status_path = Path(OUTPUT_ROOT) / 'official_full_status.json'
with open(status_path) as f:
    status = json.load(f)
assert status['status'] == 'dry_run', status['status']
assert status['training_config']['a100_ready'] is True
assert len(status['jobs']) == 12, len(status['jobs'])
print('Dry-run validado: 12 jobs planificados, A100 config lista, status oficial escrito.')


### 7.2 Entrenamiento completo 75 episodios

Ejecuta 12 corridas secuenciales: HAPPO, MASAC, MATD3 y MAAC para E1/E2/E3. Usa `--skip-completed`, por lo que si Colab se desconecta puedes reejecutar esta celda y continuara desde los artefactos completos.


In [ ]:
# ── 7.2  Lanzar entrenamiento completo recuperable ─────────────────────────
LAUNCH_FULL_TRAINING = True

if LAUNCH_FULL_TRAINING:
    train_cmd = launcher_base_args() + ['--skip-completed']
    run_cmd(train_cmd)
else:
    print('LAUNCH_FULL_TRAINING=False; no se lanzo entrenamiento.')
    print('Cambia a True para ejecutar 75 episodios en A100.')


### 7.3 Monitor visible manual

Puedes ejecutar esta celda despues del entrenamiento, o tras reabrir el notebook, para ver el ultimo estado guardado. Durante el entrenamiento, el launcher imprime snapshots visibles cada 30 segundos.


In [ ]:
# ── 7.3  Monitor visible en notebook ────────────────────────────────────────
monitor_once()


In [ ]:
# ── 7.4  Resumen global de jobs y artefactos ────────────────────────────────
import json, glob, os
from pathlib import Path

status_path = Path(OUTPUT_ROOT) / 'official_full_status.json'
if not status_path.exists():
    raise FileNotFoundError(f'No existe status oficial: {status_path}')

with open(status_path) as f:
    official_status = json.load(f)

print('=' * 72)
print('  RESUMEN DE ENTRENAMIENTO — ESTADO OFICIAL')
print('=' * 72)
print('Status:', official_status.get('status'))
print('Output:', official_status.get('output_root'))

jobs = official_status.get('jobs', [])
completed = [j for j in jobs if j.get('exit_code') == 0 and not j.get('planned_only')]
failed = [j for j in jobs if j.get('exit_code') not in (None, 0)]
planned = [j for j in jobs if j.get('planned_only')]
print(f'Jobs completados: {len(completed)} | fallidos: {len(failed)} | planificados dry-run: {len(planned)}')
for job in jobs:
    if job.get('planned_only'):
        continue
    state = 'OK' if job.get('exit_code') == 0 else ('RUNNING' if job.get('completed_at') is None else 'FAILED')
    print(f"  {job.get('name','?').upper():<6} {job.get('scenario','?')} -> {state} attempt={job.get('attempt', 0)}")

n_json  = len(glob.glob(f'{OUTPUT_ROOT}/**/*.json', recursive=True))
n_csv   = len(glob.glob(f'{OUTPUT_ROOT}/**/*.csv', recursive=True))
n_png   = len(glob.glob(f'{OUTPUT_ROOT}/**/*.png', recursive=True))
n_ckpt  = len(glob.glob(f'{OUTPUT_ROOT}/**/*.pt', recursive=True))
print(f'\nArtefactos: {n_json} JSON · {n_csv} CSV · {n_png} PNG · {n_ckpt} checkpoints .pt')


## Sección 8: Análisis de resultados y KPIs

### Estructura de artefactos (algorithm-first)
```
{OUTPUT_ROOT}/
  happo/
    E1_seed_0/data/results.json  timeseries.csv  training_summary.json
    E2_seed_0/data/results.json  ...
    E3_seed_0/data/results.json  ...
  masac/ matd3/ maac/  → misma estructura
  logs/  happo_E1.log  masac_E1.log  ...
  figures/  evaluation/
```


In [ ]:
# ── 8.1  Cargar todos los results.json ──────────────────────────────────────
import json, os, glob
import pandas as pd
import numpy as np

def load_all_results(output_root: str) -> pd.DataFrame:
    records = []
    # Layout algorithm-first: {output_root}/{algo}/{scenario}_seed_0/data/results.json
    for fp in sorted(glob.glob(f"{output_root}/*/*/data/results.json", recursive=False)):
        parts = Path(fp).parts
        algo_idx  = next(i for i,p in enumerate(parts) if p == Path(output_root).name) + 1
        algo      = parts[algo_idx] if algo_idx < len(parts) else "?"
        sc_seed   = parts[algo_idx + 1] if algo_idx+1 < len(parts) else "?"
        scenario  = sc_seed.split("_seed_")[0] if "_seed_" in sc_seed else sc_seed
        try:
            with open(fp) as f:
                data = json.load(f)
            # KPIs are nested under citylearn_v3_report.all_values, not at root level
            all_v = data.get("citylearn_v3_report", {}).get("all_values", {})
            records.append({
                "algorithm":                 algo.upper(),
                "scenario":                  scenario,
                "peak_average":              all_v.get("peak_average",                  np.nan),
                "ramping_average":           all_v.get("ramping_average",               np.nan),
                "one_minus_load_factor":     all_v.get("one_minus_load_factor_average", np.nan),
                "carbon_emissions":          all_v.get("carbon_emissions",              np.nan),
                "electricity_cost":          all_v.get("electricity_cost",              np.nan),
                "ev_departure_success_rate": all_v.get("ev_departure_success_rate",     np.nan),
                "pv_self_consumption_ratio": all_v.get("pv_self_consumption_ratio",     np.nan),
            })
        except Exception as e:
            print(f"  ⚠️  {fp}: {e}")
    return pd.DataFrame(records)

from pathlib import Path
df_results = load_all_results(OUTPUT_ROOT)

if df_results.empty:
    print("⚠️  Sin results.json todavía — ejecuta el entrenamiento primero.")
    print("   (Referencia v4: MATD3 KW p=0.0459, Score global 0.7445)")
else:
    pd.set_option("display.float_format", "{:.4f}".format)
    print(f"✅  {len(df_results)} corridas cargadas\n")
    print(df_results.to_string(index=False))
    os.makedirs(f"{OUTPUT_ROOT}/evaluation", exist_ok=True)
    df_results.to_csv(f"{OUTPUT_ROOT}/evaluation/all_kpis.csv", index=False)


In [ ]:
# ── 8.2  Curvas de convergencia (timeseries.csv, por episodio) ───────────────
import matplotlib.pyplot as plt, glob, pandas as pd
from pathlib import Path

ts_data = {}
for fp in sorted(glob.glob(f"{OUTPUT_ROOT}/*/*/data/timeseries.csv")):
    parts = Path(fp).parts
    root_idx = next(i for i,p in enumerate(parts) if p == Path(OUTPUT_ROOT).name)
    algo     = parts[root_idx + 1].upper()
    sc_seed  = parts[root_idx + 2]
    sc       = sc_seed.split("_seed_")[0] if "_seed_" in sc_seed else sc_seed
    try:
        ts_data[f"{algo}_{sc}"] = pd.read_csv(fp)
    except Exception:
        pass

if ts_data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    CLR = {"HAPPO":"#3b82f6","MASAC":"#a21caf","MATD3":"#16a34a","MAAC":"#d97706"}
    for ax, sc in zip(axes, ["E1", "E2", "E3"]):
        for key, df in ts_data.items():
            if f"_{sc}" in key:
                alg = key.replace(f"_{sc}", "")
                if "episode" in df.columns and "reward_mean" in df.columns:
                    # Aggregate step-level timeseries to episode-level mean reward
                    ep_df = df.groupby("episode")["reward_mean"].mean().reset_index()
                    smoothed = ep_df["reward_mean"].rolling(2, min_periods=1).mean()
                    ax.plot(ep_df["episode"], smoothed,
                            label=alg, color=CLR.get(alg, "gray"), lw=2, alpha=0.85)
        ax.set_title(f"Escenario {sc}", fontweight="bold")
        ax.set_xlabel("Episodio"); ax.set_ylabel("Reward medio por episodio (smoothed)")
        ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_facecolor("#f8fafc")
    fig.suptitle("Convergencia — 4 Algoritmos × 3 Escenarios", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/evaluation/convergencia.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅  {OUTPUT_ROOT}/evaluation/convergencia.png")
else:
    print("Sin timeseries disponibles.")


## Sección 9: Evaluación estadística — Selección del mejor MADRL

Protocolo idéntico al análisis oficial:
1. **Shapiro-Wilk** — normalidad por algoritmo
2. **Kruskal-Wallis** — diferencia global (4 grupos)
3. **Mann-Whitney U** — pares con effect size (Cliff's δ)
4. **Ranking global** — score ponderado por escenario


In [ ]:
# ── 9.1  Suite de pruebas estadísticas ──────────────────────────────────────
from scipy import stats
import itertools, json, os
import numpy as np, pandas as pd

SCENARIO_WEIGHTS = {
    "E1": {"peak_average": 0.50, "carbon_emissions": 0.25, "electricity_cost": 0.25},
    "E2": {"peak_average": 0.25, "carbon_emissions": 0.50, "electricity_cost": 0.25},
    "E3": {"peak_average": 0.25, "carbon_emissions": 0.25, "electricity_cost": 0.50},
}
INVERT = {"peak_average", "carbon_emissions", "electricity_cost"}  # menor = mejor

def cliff_delta(x, y):
    n1, n2 = len(x), len(y)
    d = sum(1 for a in x for b in y if a>b) - sum(1 for a in x for b in y if a<b)
    return d / (n1 * n2)

def build_scores(df: pd.DataFrame) -> dict:
    algorithms = sorted(df["algorithm"].unique())
    scores = {a: [] for a in algorithms}
    for sc, weights in SCENARIO_WEIGHTS.items():
        sub = df[df["scenario"] == sc].copy()
        if sub.empty:
            continue
        norm_cols = []
        w_arr = []
        for kpi, w in weights.items():
            if kpi not in sub.columns:
                continue
            vals = sub[kpi].astype(float)
            rng  = vals.max() - vals.min()
            nrm  = (vals - vals.min()) / rng if rng > 0 else pd.Series(0.5, index=vals.index)
            sub[f"{kpi}_n"] = 1 - nrm if kpi in INVERT else nrm
            norm_cols.append(f"{kpi}_n")
            w_arr.append(w)
        w_arr = np.array(w_arr) / sum(w_arr)
        sub["score"] = sum(sub[nc] * wt for nc, wt in zip(norm_cols, w_arr))
        for a in algorithms:
            v = sub[sub["algorithm"]==a]["score"].values
            if len(v) > 0:
                scores[a].append(float(v[0]))
    return {a: np.array(v) for a, v in scores.items() if v}

stat_results = {}
if not df_results.empty:
    score_arrays = build_scores(df_results)
    algorithms   = sorted(score_arrays.keys())

    # 1. Shapiro-Wilk
    print("1. SHAPIRO-WILK")
    for a, arr in score_arrays.items():
        if len(arr) >= 3:
            s, p = stats.shapiro(arr)
            print(f"  {a:<6}: W={s:.4f} p={p:.4f}  {'NORMAL' if p>0.05 else 'no normal'}")
        else:
            print(f"  {a:<6}: muestras insuficientes")

    # 2. Kruskal-Wallis
    print("\n2. KRUSKAL-WALLIS")
    groups = [score_arrays[a] for a in algorithms if len(score_arrays.get(a,[])) > 0]
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        sig = p < 0.05
        print(f"  H={h:.4f}  p={p:.4f}  → {'SIGNIFICATIVO ✅' if sig else 'No significativo'}")
        stat_results["kruskal_wallis"] = {"H": float(h), "p": float(p), "significant": sig}

    # 3. Mann-Whitney U
    print("\n3. MANN-WHITNEY U (pairwise + Cliff δ)")
    mwu = {}
    for a1, a2 in itertools.combinations(algorithms, 2):
        arr1, arr2 = score_arrays.get(a1, np.array([])), score_arrays.get(a2, np.array([]))
        if len(arr1)<1 or len(arr2)<1: continue
        try:
            s, p = stats.mannwhitneyu(arr1, arr2, alternative="two-sided")
            d = cliff_delta(arr1.tolist(), arr2.tolist())
            winner = a1 if arr1.mean() > arr2.mean() else a2
            mwu[f"{a1}_vs_{a2}"] = {"p": float(p), "cliff_delta": float(d), "winner": winner}
            print(f"  {a1} vs {a2}: p={p:.4f} {'✅' if p<0.05 else ''}  δ={d:.3f}  ▶ {winner}")
        except Exception as e:
            print(f"  {a1} vs {a2}: {e}")
    stat_results["mann_whitney_u"] = mwu

    # 4. Ranking
    print("\n4. RANKING GLOBAL")
    ranking = sorted(
        [{"algorithm": a, "mean_score": float(v.mean())} for a, v in score_arrays.items()],
        key=lambda x: -x["mean_score"],
    )
    for i, r in enumerate(ranking, 1):
        print(f"  {i}. {r['algorithm']:<6}  {r['mean_score']:.4f} {'★ Ganador' if i==1 else ''}")
    stat_results["ranking"]   = ranking
    stat_results["best_madrl"] = ranking[0]["algorithm"] if ranking else "N/A"

    os.makedirs(f"{OUTPUT_ROOT}/evaluation", exist_ok=True)
    with open(f"{OUTPUT_ROOT}/evaluation/statistical_analysis.json", "w") as f:
        json.dump(stat_results, f, indent=2, default=str)
    print(f"\n✅  {OUTPUT_ROOT}/evaluation/statistical_analysis.json")
else:
    print("⚠️  Sin datos — referencia oficial v4: MATD3 mejor (KW p=0.0459)")


In [ ]:
# ── 10.  Resumen final de la sesión Colab ───────────────────────────────────
import json, glob, os
from datetime import datetime

print("=" * 65)
print("  RESUMEN FINAL — MADRL CityLearn v3 · Colab A100")
print("=" * 65)
print(f"  Output root : {OUTPUT_ROOT}")
print(f"  Timestamp   : {TIMESTAMP}")
print(f"  Modo        : {'QUICK_TEST' if QUICK_TEST else 'FULL TRAINING (75 ep)'}")

n_json = len(glob.glob(f"{OUTPUT_ROOT}/**/*.json",  recursive=True))
n_csv  = len(glob.glob(f"{OUTPUT_ROOT}/**/*.csv",   recursive=True))
n_png  = len(glob.glob(f"{OUTPUT_ROOT}/**/*.png",   recursive=True))
n_ckpt = len(glob.glob(f"{OUTPUT_ROOT}/**/*.pt",    recursive=True))
print(f"\n  Artefactos : {n_json} JSON · {n_csv} CSV · {n_png} PNG · {n_ckpt} .pt")

if stat_results and "ranking" in stat_results:
    print("\n  RANKING FINAL:")
    for i, r in enumerate(stat_results["ranking"], 1):
        mark = " ★" if i == 1 else ""
        print(f"    {i}. {r['algorithm']:<6} {r['mean_score']:.4f}{mark}")
    kw = stat_results.get("kruskal_wallis", {})
    if kw:
        print(f"  KW: p={kw.get('p','?')} ({'✅' if kw.get('significant') else ''})")
else:
    print("\n  Referencia oficial v4:")
    print("    1. MATD3  0.7445 ★")
    print("    2. MASAC  ~0.73")
    print("    3. MAAC   ~0.72")
    print("    4. HAPPO  ~0.70")
    print("    KW p=0.0459 ✅")

# Escribir session summary JSON
summary = {
    "timestamp":        TIMESTAMP,
    "output_root":      OUTPUT_ROOT,
    "mode":             "quick_test" if QUICK_TEST else "full_training",
    "episodes":         EPISODES,
    "episode_steps":    EPISODE_STEPS,
    "num_env_steps":    NUM_ENV_STEPS,
    "algorithms":       ALGORITHMS,
    "scenarios":        SCENARIOS,
    "a100_tuning": {
        "happo_hidden":         384,
        "masac_buffer_size":    20,
        "masac_critic_batch":   64,
        "masac_max_buf_gib":    20,
        "matd3_batch_size":     512,
        "matd3_buffer_size":    6000,
        "maac_batch_size":      512,
        "maac_buffer_length":   100000,
    },
    "artifacts": {"json": n_json, "csv": n_csv, "png": n_png, "pt": n_ckpt},
    "statistical_analysis": stat_results if stat_results else "run training first",
}
with open(f"{OUTPUT_ROOT}/colab_session_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n  ✅  Resumen: {OUTPUT_ROOT}/colab_session_summary.json")
print("=" * 65)


## Proximos pasos

1. Si Colab se desconecta, vuelve a ejecutar configuracion inicial y la celda **7.2**; `--skip-completed` evita repetir jobs completos.
2. Para revisar estado sin entrenar, ejecuta `CityLearn/scripts/colab_a100_live_monitor.py --output-root <OUTPUT_ROOT> --once`.
3. Para evidencia de tesis, conserva `official_full_status.json`, `official_full_manifest.json`, `training_summary.json`, `results.json`, `timeseries.csv`, checkpoints y `colab_session_summary.json`.
4. Para validez estadistica fuerte, repetir con seeds adicionales cuando haya presupuesto de GPU.

Repositorio: [Mac-Tapia/MADRLCitytleranflexresdr](https://github.com/Mac-Tapia/MADRLCitytleranflexresdr)
Contacto: mac.tapia.c@uni.pe · Universidad Nacional de Ingenieria - UNI · 2026
